# British Airways Review Analysis

This notebook covers importing, exploring, and cleaning the British Airways customer review dataset (`British_Airway_Review.csv`) scraped from Skytrax. The goal is to prepare a clean, analysis-ready dataset and document the reasoning behind each cleaning decision along the way.

**Dataset:** 2,500 reviews with 8 columns — review text, date, country, seat type, recommendation, star rating, route, and traveller type.

## Contents
1. Setup and Data Import
2. Initial Exploration (EDA)
3. Cleaning the `reviews` Column
4. Cleaning the `date` Column
5. Investigating the `stars` Column
6. Summary and Next Steps

## 1. Setup and Data Import

Importing the core libraries and loading the raw CSV into a DataFrame.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../data/raw/British_Airway_Review.csv')
print(df.head())
print(df.shape)

                                             reviews             date  \
0  ✅ Trip Verified |  I had the most fantastic BA...  1st August 2023   
1  ✅ Trip Verified |  Couldn’t book in online. Ar...   31st July 2023   
2  ✅ Trip Verified | London Heathrow to Mumbai in...   31st July 2023   
3  ✅ Trip Verified | Keflavík, Iceland to London ...   31st July 2023   
4  ✅ Trip Verified | Terrible Experience with Bri...   29th July 2023   

          country       seat_type recommended  stars                  route  \
0       Hong Kong  Business Class         yes      5  Heathrow to Las Vegas   
1  United Kingdom   Economy Class          no      3       Rome to Heathrow   
2         Iceland  Business Class         yes      3      Gatwick to Venice   
3         Iceland  Business Class         yes      5       London to Luanda   
4          Canada   Economy Class          no      5     Denver to Heathrow   

  type_of_traveller  
0    Family Leisure  
1      Solo Leisure  
2      Solo Leisure 

## 2. Initial Exploration (EDA)

Before cleaning anything, it's worth understanding the shape and quality of the data: column types, missing values, duplicate rows, and a look at the raw text of the `reviews` column itself.

**Findings:** no missing values and no duplicate rows across all 2,500 records. However, the `reviews` column contains an inconsistent verification-status prefix (e.g. `✅ Trip Verified |`, `Not Verified |`) mixed in with the actual review text, and some older reviews have no prefix at all.

In [3]:
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   reviews            2500 non-null   str  
 1   date               2500 non-null   str  
 2   country            2500 non-null   str  
 3   seat_type          2500 non-null   str  
 4   recommended        2500 non-null   str  
 5   stars              2500 non-null   int64
 6   route              2500 non-null   str  
 7   type_of_traveller  2500 non-null   str  
dtypes: int64(1), str(7)
memory usage: 156.4 KB
None


In [4]:
print(df.isna().sum())

reviews              0
date                 0
country              0
seat_type            0
recommended          0
stars                0
route                0
type_of_traveller    0
dtype: int64


In [5]:
print(df.duplicated().sum())

0


In [6]:
print(df['reviews'].head(10))

0    ✅ Trip Verified |  I had the most fantastic BA...
1    ✅ Trip Verified |  Couldn’t book in online. Ar...
2    ✅ Trip Verified | London Heathrow to Mumbai in...
3    ✅ Trip Verified | Keflavík, Iceland to London ...
4    ✅ Trip Verified | Terrible Experience with Bri...
5    ✅ Trip Verified | An airline that lives in the...
6    ✅ Trip Verified |  Check-in Desk rude and dism...
7    ✅ Trip Verified | I chose British Airways espe...
8    Not Verified | I booked Premium Economy from I...
9    ✅ Trip Verified | A simple story with an unfor...
Name: reviews, dtype: str


## 3. Cleaning the `reviews` Column

### 3.1 Extracting Verification Status

The `reviews` column mixes a verification tag ("Trip Verified", "Not Verified", etc.) with the actual review text, separated by a `|`. Across the dataset's history this tag appears in several different phrasings (`✅ Trip Verified`, `Not Verified`, `❎ Unverified`), and pre-2016 reviews have no tag at all.

To capture this properly, a function (`review_verification_status`) classifies each row into `True` (verified), `False` (explicitly not verified), or a missing value (`pd.NA`, for untagged rows) — stored in a new `is_verified` column using pandas' nullable boolean dtype, which allows a genuine "unknown" state alongside `True`/`False` rather than forcing untagged rows into `False`.

In [7]:
df['is_verified'] = df['reviews'].str.startswith('✅ Trip Verified ')
filtered_df = df[df['is_verified'] == False]
print(filtered_df['reviews'].str.count('\\|').value_counts())

reviews
1    1001
0     408
Name: count, dtype: int64


In [8]:
def review_verification_status(review_string):
    if '|' not in review_string:
        return pd.NA
    elif 'Not' in review_string.split('|')[0]:
        return False
    elif '✅' in review_string.split('|')[0]:
        return True
    elif '❎' in review_string.split('|')[0]:
        return False
    else:
        return pd.NA


print(review_verification_status("Not Verified | some review"))
print(review_verification_status("✅ Trip Verified | some review"))
print(review_verification_status("❎ Unverified | some review"))

False
True
False


In [9]:
df['is_verified'] = df['reviews'].apply(review_verification_status).astype('boolean')


In [10]:
print(df.head())

                                             reviews             date  \
0  ✅ Trip Verified |  I had the most fantastic BA...  1st August 2023   
1  ✅ Trip Verified |  Couldn’t book in online. Ar...   31st July 2023   
2  ✅ Trip Verified | London Heathrow to Mumbai in...   31st July 2023   
3  ✅ Trip Verified | Keflavík, Iceland to London ...   31st July 2023   
4  ✅ Trip Verified | Terrible Experience with Bri...   29th July 2023   

          country       seat_type recommended  stars                  route  \
0       Hong Kong  Business Class         yes      5  Heathrow to Las Vegas   
1  United Kingdom   Economy Class          no      3       Rome to Heathrow   
2         Iceland  Business Class         yes      3      Gatwick to Venice   
3         Iceland  Business Class         yes      5       London to Luanda   
4          Canada   Economy Class          no      5     Denver to Heathrow   

  type_of_traveller  is_verified  
0    Family Leisure         True  
1      Solo Leis

### 3.2 Stripping the Verification Prefix from Review Text

With the verification status captured separately in `is_verified`, the tag prefix itself is no longer needed inside `reviews`. A second function, `extract_review_text`, removes everything up to and including the `|` for tagged rows and leaves untagged rows unchanged, so the column contains clean review text only. This was verified against both tagged rows (prefix removed) and untagged rows (left untouched).

In [11]:
def extract_review_text(review_string):
    if '|' not in review_string:
        return review_string
    else:
        return review_string.split('|')[1].strip()

In [12]:
df['reviews'] = df['reviews'].apply(extract_review_text)
print(df['reviews'].head(10))

0    I had the most fantastic BA Flight today. The ...
1    Couldn’t book in online. Arrived at check in t...
2    London Heathrow to Mumbai in a Boeing 787-8 in...
3    Keflavík, Iceland to London Heathrow on an A32...
4    Terrible Experience with British Airways. I bo...
5    An airline that lives in their past glory and ...
6    Check-in Desk rude and dismissive. Flight late...
7    I chose British Airways especially because I w...
8    I booked Premium Economy from INV to LAX (via ...
9    A simple story with an unfortunate outcome tha...
Name: reviews, dtype: str


In [13]:
filtered_df = df[df['is_verified'].isna()]
print(filtered_df['reviews'].head(10))

1187    Two regular an uneventful flights. Curiously e...
1828    Flew London Gatwick to Tampa with British Airw...
1832    Frankfurt to Cape Town via London Heathrow. Th...
1841    Bangkok to Manchester via London. I used to fl...
1842    The return flight from St Lucia started well. ...
1844    London Heathrow to Hong Kong. We have not flow...
1857    Appalling food, especially on the Orlando-Gatw...
1863    Flight from LGW to St Lucia got off to a puzzl...
1865    London Heathrow to Cape Town with British Airw...
1867    Shocking. There is nothing good to say about t...
Name: reviews, dtype: str


## 4. Cleaning the `date` Column

The `date` column was stored as plain text in formats like "1st August 2023", including an ordinal suffix (st/nd/rd/th) on the day. A single string like this parses fine with `pd.to_datetime()` on its own, but converting the whole column at once initially failed:

```
ValueError: time data "29th July 2023" doesn't match format "%dst %B %Y"
```

This happens because pandas infers **one** format from the first value in the column (hardcoding the suffix it saw, e.g. "st") and then tries to apply that same format to every row — which breaks as soon as a row needs a different suffix ("th", "nd", "rd").

The fix was to pass `format='mixed'` to `pd.to_datetime()`, which infers the format for each value individually rather than assuming one format for the whole column. The result was verified with `.isna().sum()`, confirming 0 rows failed to parse, and the column was converted in place to a proper `datetime64` dtype.

In [14]:
print(df['date'].sample(10))

374       10th January 2022
818            1st May 2019
1852     11th December 2016
1699         9th March 2017
219        6th October 2022
696        28th August 2019
676     18th September 2019
848         19th March 2019
1289     24th November 2017
2039    11th September 2016
Name: date, dtype: str


In [15]:
print(pd.to_datetime('1st August 2023'))
print(pd.to_datetime('1 August 2023'))

2023-08-01 00:00:00
2023-08-01 00:00:00


In [16]:
df['date'] = pd.to_datetime(
    df['date'],
    format='mixed',
    dayfirst=True
)
print(df['date'].head())

0   2023-08-01
1   2023-07-31
2   2023-07-31
3   2023-07-31
4   2023-07-29
Name: date, dtype: datetime64[us]


In [17]:
print(df['date'].isna().sum())
print(df.head(10))

0
                                             reviews       date  \
0  I had the most fantastic BA Flight today. The ... 2023-08-01   
1  Couldn’t book in online. Arrived at check in t... 2023-07-31   
2  London Heathrow to Mumbai in a Boeing 787-8 in... 2023-07-31   
3  Keflavík, Iceland to London Heathrow on an A32... 2023-07-31   
4  Terrible Experience with British Airways. I bo... 2023-07-29   
5  An airline that lives in their past glory and ... 2023-07-26   
6  Check-in Desk rude and dismissive. Flight late... 2023-07-26   
7  I chose British Airways especially because I w... 2023-07-25   
8  I booked Premium Economy from INV to LAX (via ... 2023-07-24   
9  A simple story with an unfortunate outcome tha... 2023-07-21   

          country        seat_type recommended  stars                  route  \
0       Hong Kong   Business Class         yes      5  Heathrow to Las Vegas   
1  United Kingdom    Economy Class          no      3       Rome to Heathrow   
2         Iceland   

## 5. Investigating the `stars` Column

`stars` only ever takes the values 1, 3, 5, 7, or 9 — no even numbers appear at all, which is unusual for a rating scale.

Checking a related, similarly-scraped British Airways review dataset on Kaggle (which describes `stars` as "typically on a scale of 1 to 5") revealed a discussion thread where the dataset's own creator explained the cause: while scraping the reviews, the ratings ended up limited to (1, 3, 5, 7, 9) on what was actually meant to be a 1-to-10 scale, because "the variable lengths needed to match exactly" — i.e. a scraping bug silently dropped the even values.

A follow-up comment in that same thread speculated that this could be remapped onto a 1-to-5 scale (e.g. 7 → 4), but the dataset creator never confirmed this, so it remains an unverified guess rather than a documented fact.

**Decision:** `stars` is left unchanged. Recoding the values based on an unconfirmed assumption risks quietly introducing incorrect data into the dataset. Since this is a known scraping limitation rather than a meaningful pattern, it's documented here instead of "fixed" — any future analysis using `stars` should treat it as a genuine but incomplete rating scale (values 2, 4, 6, 8, and 10 are structurally absent, not zero-count by chance).

In [18]:
print(df['stars'].value_counts().sort_index())

stars
1     227
3    1137
5     682
7     227
9     227
Name: count, dtype: int64


## 6. Summary and Next Steps

**Completed so far:**
- Imported the raw dataset and confirmed no missing values or duplicate rows.
- Extracted verification status into a new `is_verified` column (nullable boolean: `True` / `False` / `<NA>`).
- Cleaned the `reviews` column by removing the verification-tag prefix.
- Parsed the `date` column into a proper `datetime64` dtype using `format='mixed'`, verifying 0 rows failed to parse.
- Investigated the `stars` column's odd-only values, traced the cause to a known scraping limitation in the source dataset, and made the deliberate decision to leave it unchanged rather than guess at a recoding.

**Core cleaning is now complete.** Possible next steps: derived features (e.g. review length, date-based features), further exploratory analysis, and exporting the cleaned dataset to `data/processed/`.

In [19]:
df.to_csv(
    '../data/processed/cleaned_dataset.csv',
    index=False
)